# Hyperparameter tuning (Optuna) — run once before `mimic_ftt_early_warning.ipynb`
**Author:** Nabil Walid Rafi — Department of ICT, FST, Bangladesh University of Professionals

Uses exactly the same cohort, features and patient-level split as the main notebook (shared cells). Every model is
tuned on the **validation patients only** (10% of the training patients); the test patients are never loaded into any
objective. Objective: validation macro-F1 (four NEWS2 categories, class-weighted). Sampler: TPE (seed 42); the
published default configuration is evaluated as the first trial of every study; FT-Transformer trials that fall
behind are stopped early (median pruner).

Output: `best_params.json` (+ trial tables and figure). **Save & Run All**, then attach this notebook's output to the
main notebook (Add Input → Notebooks), together with `mimic-slim`.

Budgets (environment variables): `TUNE_TRIALS_FTT`=40, `TUNE_TRIALS_XGB`=100, `TUNE_TRIALS_LGBM`=100, `TUNE_TRIALS_RF`=30;
time caps `TUNE_MIN_*` (minutes): FT-Transformer 180, XGBoost 60, LightGBM 60, random forest 30.

## 1. Setup

In [ ]:
import os, glob, gc, copy, math, random, time, warnings
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, matplotlib.patches as mpatches, seaborn as sns
from scipy.stats import spearmanr
from sklearn.model_selection import StratifiedGroupKFold, GroupShuffleSplit
from sklearn.preprocessing import StandardScaler, QuantileTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (average_precision_score, precision_recall_curve, accuracy_score, balanced_accuracy_score, f1_score, precision_score, recall_score,
                             roc_auc_score, roc_curve, auc, confusion_matrix, classification_report)
from sklearn.utils.class_weight import compute_class_weight, compute_sample_weight
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
import lightgbm as lgbm
import shap, torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
FTT_EPOCHS   = int(os.environ.get('FTT_EPOCHS', 200))     # maximum; early stopping usually ends much sooner
FTT_PATIENCE = int(os.environ.get('FTT_PATIENCE', 16))    # epochs without validation improvement
N_SEEDS      = int(os.environ.get('N_SEEDS', 5))          # FT-Transformer seed ensemble
FTT_EMB      = os.environ.get('FTT_EMB', 'plr')           # 'plr' (periodic, Gorishniy 2022) or 'linear'
N_SEEDS_SENS = int(os.environ.get('N_SEEDS_SENS', 3))      # FT-Transformer seeds in each sensitivity analysis
RUN_SENS     = os.environ.get('RUN_SENS', '1') == '1'      # run sensitivity analyses C–E
SHORT_H      = 30                                          # sensitivity analysis E: outcome window 24–30 h
N_BOOT     = int(os.environ.get('N_BOOT', 1000))
N_PERM     = int(os.environ.get('N_PERM', 5))
OBS_H, PRED_H = 24, 48

plt.rcParams.update({'figure.dpi': 120, 'savefig.dpi': 300, 'font.family': 'DejaVu Sans',
    'axes.spines.top': False, 'axes.spines.right': False, 'axes.titlesize': 11, 'axes.labelsize': 10,
    'xtick.labelsize': 9, 'ytick.labelsize': 9, 'axes.edgecolor': '#52514e', 'text.color': '#0b0b0b',
    'xtick.color': '#52514e', 'ytick.color': '#52514e', 'legend.frameon': False})
CLASSES     = ['Low', 'Low–medium', 'Medium', 'High']        # official NEWS2 clinical-risk categories
CLASS_COLORS = ['#86b6ef', '#3987e5', '#1c5cab', '#0d366b']    # ordinal blue ramp (validated)
C_BLUE, C_ORANGE, C_AQUA, C_GREY = '#2a78d6', '#eb6834', '#1baf7a', '#9a9893'

FIG_DIR = os.environ.get('FIG_DIR', '.'); os.makedirs(FIG_DIR, exist_ok=True)
def save_fig(fig, name):
    fig.savefig(f'{FIG_DIR}/{name}.png', dpi=300, bbox_inches='tight', facecolor='white')
    fig.savefig(f'{FIG_DIR}/{name}.pdf', bbox_inches='tight', facecolor='white'); print(f'✅ {name}.png/.pdf')
def save_table(df, name):
    df.to_csv(f'{FIG_DIR}/{name}.csv', index=False); print(f'✅ {name}.csv')

# ── Official NEWS2 scoring (RCP 2017; SpO2 scale 1) ───────────────────────────
VITALS = ['RR', 'SpO2', 'HR', 'SBP', 'Temp']
def news2_parts(rr, spo2, o2, sbp, hr, cons, temp):
    rr, spo2, sbp, hr, temp = (np.asarray(x, float) for x in (rr, spo2, sbp, hr, temp))
    cons, o2 = np.asarray(cons, float), np.asarray(o2, float)
    s = {
      'RR':   np.select([rr <= 8, rr <= 11, rr <= 20, rr <= 24, rr > 24], [3, 1, 0, 2, 3], 0),
      'SpO2': np.select([spo2 <= 91, spo2 <= 93, spo2 <= 95], [3, 2, 1], 0),
      'O2':   np.where(o2 == 1, 2, 0),
      'SBP':  np.select([sbp <= 90, sbp <= 100, sbp <= 110, sbp <= 219, sbp >= 220], [3, 2, 1, 0, 3], 0),
      'HR':   np.select([hr <= 40, hr <= 50, hr <= 90, hr <= 110, hr <= 130, hr > 130], [3, 1, 0, 1, 2, 3], 0),
      'Cons': np.where(cons > 0, 3, 0),
      'Temp': np.select([temp <= 35.0, temp <= 36.0, temp <= 38.0, temp <= 39.0, temp > 39.0], [3, 1, 0, 1, 2], 0)}
    total = sum(s.values()); red = np.max(np.stack(list(s.values())), axis=0) == 3
    return total.astype(int), red
def news2_category(total, red):
    # High ≥7; Medium 5–6; Low–medium = a single parameter scoring 3; Low otherwise (0–4)
    return np.select([total >= 7, total >= 5, red], [3, 2, 1], 0)
print(f'device={device} | FT-T max epochs={FTT_EPOCHS}, patience={FTT_PATIENCE}, seeds={N_SEEDS}, embedding={FTT_EMB} | bootstrap={N_BOOT}')


## 2. Locate inputs (slim parquet preferred; raw CSV fallback)

In [ ]:
DATA_ROOT = os.environ.get('DATA_ROOT', '/kaggle/input')
def find(*patterns):
    for p in patterns:
        hits = sorted(glob.glob(os.path.join(DATA_ROOT, '**', p), recursive=True)) + sorted(glob.glob(p))
        if hits: return hits[0]
    return None
P_ICU = find('icustays.parquet', 'icustays.csv*')
P_ADM = find('admissions.parquet', 'admissions.csv*')
P_CE  = find('chartevents_vitals_48h.parquet', 'chartevents_vitals_filtered.csv*')
P_PAT = find('patients.parquet')            # optional extras (skipped with a warning if absent)
P_LAB = find('labs_48h.parquet')
P_PRO = find('procedures_vent.parquet')
P_INP = find('inputs_vaso.parquet')
for k, v in [('icustays', P_ICU), ('admissions', P_ADM), ('chartevents', P_CE), ('patients', P_PAT),
             ('labs', P_LAB), ('ventilation', P_PRO), ('vasopressors', P_INP)]: print(f'{k:12s}: {v}')
for k, v in [('patients', P_PAT), ('labs', P_LAB), ('ventilation', P_PRO), ('vasopressors', P_INP)]:
    if not v: print(f'⚠ {k} not found — those features will be skipped (re-run make_mimic_slim to add them)')
assert P_ICU and P_ADM and P_CE, 'Attach the private mimic-slim dataset (see make_mimic_slim.ipynb).'


## 3. Load, harmonise and tag time windows

In [ ]:
rd = lambda p, **kw: pd.read_parquet(p) if p.endswith('.parquet') else pd.read_csv(p, **kw)
icu = rd(P_ICU)
adm = rd(P_ADM)
adm = adm[[c for c in ['subject_id', 'hadm_id', 'admission_type', 'deathtime', 'hospital_expire_flag'] if c in adm.columns]]
stays = icu.merge(adm, on=['subject_id', 'hadm_id'], how='left')
for c in ['intime', 'outtime', 'deathtime']: stays[c] = pd.to_datetime(stays[c], errors='coerce')

ce = rd(P_CE)
HAS_DEVICE_TEXT = 'value' in ce.columns
ce['charttime'] = pd.to_datetime(ce['charttime'], errors='coerce')
ce = ce.merge(stays[['stay_id', 'intime']], on='stay_id', how='inner')
ce['h'] = (ce['charttime'] - ce['intime']).dt.total_seconds() / 3600
ce = ce[(ce['h'] >= 0) & (ce['h'] < PRED_H)].sort_values(['stay_id', 'charttime'])
ce['window'] = np.where(ce['h'] < OBS_H, 'obs', 'pred')

ITEM = {220210: 'RR', 220277: 'SpO2', 220045: 'HR', 220179: 'SBP', 220050: 'SBP', 223762: 'Temp', 223761: 'TempF',
        220739: 'GCS_eye', 223900: 'GCS_verbal', 223901: 'GCS_motor', 220181: 'MAP', 220052: 'MAP',
        226732: 'O2_device', 223834: 'O2_flow'}   # SBP: non-invasive or arterial; MAP: non-invasive or arterial
ce['var'] = ce['itemid'].map(ITEM); ce = ce.dropna(subset=['var'])
f = ce['var'] == 'TempF'
ce.loc[f, 'valuenum'] = (ce.loc[f, 'valuenum'] - 32) * 5 / 9; ce.loc[f, 'var'] = 'Temp'

# Consciousness from GCS eye opening → ACVPU-style code: 4 spontaneous→0 (alert), 3→2 (voice), 2→3 (pain), 1→4 (unresponsive)
g = ce['var'] == 'GCS_eye'
ce.loc[g, 'valuenum'] = ce.loc[g, 'valuenum'].round().map({4: 0, 3: 2, 2: 3, 1: 4})
ce.loc[g, 'var'] = 'Cons'

# Supplemental oxygen: a charted device other than 'None', or O2 flow > 0 L/min
o = ce['var'] == 'O2_device'
if HAS_DEVICE_TEXT:
    txt = ce.loc[o, 'value'].astype('string').str.strip().str.lower()
    ce.loc[o, 'valuenum'] = np.where(txt.isin(['none', 'room air', 'ra']) | txt.isna(), 0.0, 1.0)
else:
    print('⚠ No device text in input — any charted O2 device is treated as oxygen (re-run make_mimic_slim).')
    ce.loc[o, 'valuenum'] = 1.0
fl = ce['var'] == 'O2_flow'
ce.loc[fl, 'valuenum'] = (ce.loc[fl, 'valuenum'] > 0).astype(float)
ce.loc[o | fl, 'var'] = 'O2'

VALID = {'RR': (4, 60), 'SpO2': (50, 100), 'HR': (20, 250), 'SBP': (40, 260), 'Temp': (30, 43), 'MAP': (20, 200),
         'GCS_verbal': (1, 5), 'GCS_motor': (1, 6)}
bad = np.zeros(len(ce), bool)
for v, (lo, hi) in VALID.items():
    m = (ce['var'] == v).values
    bad |= m & ~ce['valuenum'].between(lo, hi).values
ce = ce[~bad].dropna(subset=['valuenum'])
print(f'Artefacts removed: {bad.sum():,} | rows kept: {len(ce):,} | stays: {ce.stay_id.nunique():,}')


## 4. Window summaries

In [ ]:
agg = (ce.groupby(['stay_id', 'window', 'var'])['valuenum']
         .agg(['median', 'min', 'max', 'std', 'first', 'last', 'count']).reset_index())
def wide(win, stats):
    a = agg[agg['window'] == win].pivot(index='stay_id', columns='var', values=stats)
    a.columns = [f'{v}_{s}' for s, v in a.columns]; return a
obs  = wide('obs',  ['median', 'min', 'max', 'std', 'first', 'last', 'count'])
pred = wide('pred', ['median', 'max', 'count'])
for d, cols in [(obs, [f'{v}_{s}' for v in VITALS + ['Cons', 'O2', 'MAP', 'GCS_verbal', 'GCS_motor'] for s in ['median', 'min', 'max', 'std', 'first', 'last', 'count']]),
                (pred, [f'{v}_{s}' for v in VITALS + ['Cons', 'O2'] for s in ['median', 'max', 'count']])]:
    for c in cols:
        if c not in d.columns: d[c] = np.nan
c6 = ce[(ce['h'] >= OBS_H) & (ce['h'] < SHORT_H)]              # sensitivity analysis E: 24–30 h
pred6 = c6.groupby(['stay_id', 'var'])['valuenum'].agg(['median', 'max', 'count']).unstack('var')
pred6.columns = [f'{v}_{s}' for s, v in pred6.columns]
for c in [f'{v}_{s}' for v in VITALS + ['Cons', 'O2'] for s in ['median', 'max', 'count']]:
    if c not in pred6.columns: pred6[c] = np.nan
print(f'obs-window stays: {len(obs):,} | pred-window stays: {len(pred):,} | 24–30 h stays: {len(pred6):,}')
del ce, agg, c6; gc.collect()


## 5. Outcome (official NEWS2 category at 24–48 h), cohort and patient-level split
Prediction-window NEWS2 uses per-stay medians of RR, SpO₂, SBP, HR and temperature, the median consciousness
code, and supplemental oxygen if any was charted. A parameter not recorded at 24–48 h is carried forward
from 0–24 h. Death between 24 and 48 h → High.

In [ ]:
flow = {'ICU stays (MIMIC-IV v3.1)': len(stays), 'Unique patients': stays.subject_id.nunique()}
df = stays[['subject_id', 'stay_id', 'intime', 'deathtime']].merge(obs, left_on='stay_id', right_index=True)
flow['With 0–24 h vital signs'] = len(df)
df = df[df[[f'{v}_median' for v in VITALS]].notnull().sum(axis=1) >= 3]
flow['≥3 of 5 core vitals at 0–24 h'] = len(df)
hrs_death = (df['deathtime'] - df['intime']).dt.total_seconds() / 3600
df = df[~(hrs_death < OBS_H)]
flow['Alive at 24 h'] = len(df)
died = ((df['deathtime'] - df['intime']).dt.total_seconds() / 3600).between(OBS_H, PRED_H, inclusive='left').values

P = pred.reindex(df['stay_id'])
measured = P[[f'{v}_median' for v in VITALS]].notnull().sum(axis=1).values
keep = (measured >= 3) | died
df, P, died = df[keep].reset_index(drop=True), P[keep].reset_index(drop=True), died[keep]
flow['Outcome available at 24–48 h (final cohort)'] = len(df)

def locf(pcol, ocol): return P[pcol].fillna(pd.Series(df[ocol].values)).values
tot, red = news2_parts(locf('RR_median', 'RR_median'), locf('SpO2_median', 'SpO2_median'),
                       np.where(P['O2_max'].notnull(), P['O2_max'], df['O2_max'].values),
                       locf('SBP_median', 'SBP_median'), locf('HR_median', 'HR_median'),
                       np.round(locf('Cons_median', 'Cons_median')), locf('Temp_median', 'Temp_median'))
df['NEWS2_24_48'] = tot
df['y'] = news2_category(tot, red)
df.loc[died, 'y'] = 3
df['died_24_48'] = died.astype(int)

# Sensitivity analysis E: NEWS2 category over 24–30 h, within the same cohort (≥3 core vitals at 24–30 h or death in 24–30 h)
P6 = pred6.reindex(df['stay_id']).reset_index(drop=True)
died6 = ((df['deathtime'] - df['intime']).dt.total_seconds() / 3600).between(OBS_H, SHORT_H, inclusive='left').values
ok6 = (P6[[f'{v}_median' for v in VITALS]].notnull().sum(axis=1).values >= 3) | died6
def locf6(c): return P6[c].fillna(pd.Series(df[c].values)).values
t6, r6 = news2_parts(locf6('RR_median'), locf6('SpO2_median'), np.where(P6['O2_max'].notnull(), P6['O2_max'], df['O2_max'].values),
                     locf6('SBP_median'), locf6('HR_median'), np.round(locf6('Cons_median')), locf6('Temp_median'))
df['y_6h'] = np.where(ok6, news2_category(t6, r6), np.nan); df.loc[died6, 'y_6h'] = 3

sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
tr_i, te_i = next(sgkf.split(df, df['y'], groups=df['subject_id']))
df['split'] = 'train'; df.loc[te_i, 'split'] = 'test'
assert set(df.loc[df.split == 'train', 'subject_id']).isdisjoint(df.loc[df.split == 'test', 'subject_id'])
flow['Train stays / patients'] = f"{(df.split=='train').sum():,} / {df.loc[df.split=='train','subject_id'].nunique():,}"
flow['Test stays / patients']  = f"{(df.split=='test').sum():,} / {df.loc[df.split=='test','subject_id'].nunique():,}"
for k, v in flow.items(): print(f'  {k:45s}: {v:,}' if isinstance(v, (int, np.integer)) else f'  {k:45s}: {v}')
for k in range(4):
    n = (df.y == k).sum(); print(f'  {CLASSES[k]:11s}: {n:7,} ({n/len(df)*100:.1f}%)')
print(f'  deaths 24–48 h: {df.died_24_48.sum():,} | stays with a 24–30 h outcome (analysis E): {df.y_6h.notnull().sum():,}')


## 5b. Context, laboratory values and treatments (0–24 h)
Age and sex (patients), emergency admission (admissions), first ICU care unit (icustays); laboratory values from
6 h before to 24 h after ICU admission (worst value: max or min); invasive or non-invasive ventilation and
vasopressor infusion active at any time in 0–24 h. Missing sources are skipped with a warning.

In [ ]:
LAB_DEF = {50813: ('lactate', 'max'), 50912: ('creatinine', 'max'), 51006: ('bun', 'max'), 51301: ('wbc', 'max'),
           51300: ('wbc', 'max'), 51265: ('platelets', 'min'), 50885: ('bilirubin', 'max'), 51222: ('hemoglobin', 'min'),
           50882: ('bicarbonate', 'min'), 50971: ('potassium', 'max'), 50931: ('glucose', 'max'), 50983: ('sodium', 'min')}
LAB_F, CTX_F, TRT_F = [], [], []
if P_PAT:
    pat = rd(P_PAT)[['subject_id', 'gender', 'anchor_age', 'anchor_year']]
    m = df[['subject_id', 'intime']].merge(pat, on='subject_id', how='left')
    df['age'] = (m['anchor_age'] + (m['intime'].dt.year - m['anchor_year'])).values.astype(float)
    df['male'] = (m['gender'] == 'M').astype(float).values
    CTX_F += ['age', 'male']
st = stays.set_index('stay_id')
if 'admission_type' in st.columns:
    at = df['stay_id'].map(st['admission_type']).astype('string').str.upper()
    df['emergency_admission'] = at.str.contains('EMER|URGENT', na=False).astype(float).values
    CTX_F.append('emergency_admission')
if 'first_careunit' in st.columns:
    def unit_group(u):
        u = str(u)
        if 'Neuro' in u: return 'neuro'
        if 'Cardiac' in u or 'CCU' in u or 'Coronary' in u: return 'cardiac'
        if 'Medical' in u or 'MICU' in u: return 'medical'
        if 'Surgical' in u or 'SICU' in u or 'Trauma' in u: return 'surgical'
        return 'other'
    ug = df['stay_id'].map(st['first_careunit']).map(unit_group)
    for g in ['cardiac', 'surgical', 'neuro', 'other']:          # medical ICU = reference
        df[f'unit_{g}'] = (ug == g).astype(float).values; CTX_F.append(f'unit_{g}')
if P_LAB:
    lab = rd(P_LAB)
    lab = lab[(lab['hours_from_intime'] >= -6) & (lab['hours_from_intime'] < OBS_H) & (lab['valuenum'] > 0)]
    lab['name'] = lab['itemid'].map({k: v[0] for k, v in LAB_DEF.items()})
    g = lab.dropna(subset=['name']).groupby(['stay_id', 'name'])['valuenum']
    lmax, lmin = g.max(), g.min()
    for name, how in {v[0]: v[1] for v in LAB_DEF.values()}.items():
        src = (lmax if how == 'max' else lmin)
        src = src[src.index.get_level_values('name') == name].droplevel('name')
        df[f'lab_{name}'] = df['stay_id'].map(src).values; LAB_F.append(f'lab_{name}')
    df['lactate_measured'] = df['lab_lactate'].notnull().astype(float); LAB_F.append('lactate_measured')
    print('lab availability (% of stays):', {c: round(df[c].notnull().mean() * 100, 1) for c in LAB_F if c != 'lactate_measured'})
def active_0_24(path, items=None):
    e = rd(path); e = e[(e['start_h'] < OBS_H) & (e['end_h'] >= 0)]
    if items: e = e[e['itemid'].isin(items)]
    return df['stay_id'].isin(e['stay_id']).astype(float).values
if P_PRO:
    df['vent_invasive'] = active_0_24(P_PRO, [225792]); df['vent_noninvasive'] = active_0_24(P_PRO, [225794]); TRT_F += ['vent_invasive', 'vent_noninvasive']
if P_INP:
    df['vasopressor'] = active_0_24(P_INP); TRT_F.append('vasopressor')
print(f'context: {CTX_F}\nlabs: {LAB_F}\ntreatments: {TRT_F}')


## 6. Feature set (0–24 h)
Vital-sign summaries (RR, SpO₂, HR, SBP, temperature: median, min, max, SD, last, change), MAP (median, min, last),
GCS verbal and motor (min), consciousness (worst, last), supplemental O₂, NEWS2 at 0–24 h, laboratory worst values,
age, sex, emergency admission, care unit, ventilation and vasopressors. Training-patient medians impute missing
continuous values (SD/change → 0; flags → 0).

In [ ]:
for v in VITALS:
    df[f'{v}_delta'] = df[f'{v}_last'] - df[f'{v}_first']
df['Cons_worst'] = df['Cons_max']; df['O2_any'] = df['O2_max']
VITAL_F = [f'{v}_{s}' for v in VITALS for s in ['median', 'min', 'max', 'std', 'last', 'delta']]
EXTRA_VITAL_F = [c for c in ['MAP_median', 'MAP_min', 'MAP_last', 'GCS_verbal_min', 'GCS_motor_min'] if df[c].notnull().mean() > 0.05]
NEURO_O2_F = ['Cons_worst', 'Cons_last', 'O2_any', 'NEWS2_0_24']
FEATURES = VITAL_F + EXTRA_VITAL_F + NEURO_O2_F + LAB_F + CTX_F + TRT_F

tr_mask = (df.split == 'train').values
for c in [f'{v}_{s}' for v in VITALS for s in ['median', 'min', 'max', 'last']] + EXTRA_VITAL_F + [c for c in LAB_F if c.startswith('lab_')] + [c for c in ['age'] if c in df]:
    df[c] = df[c].fillna(df.loc[tr_mask, c].median())
for c in [f'{v}_{s}' for v in VITALS for s in ['std', 'delta']]: df[c] = df[c].fillna(0.0)
for c in ['Cons_worst', 'Cons_last', 'Cons_median', 'O2_any'] + [c for c in CTX_F + TRT_F if c != 'age']:
    df[c] = df[c].fillna(0)
t0, r0 = news2_parts(df['RR_median'], df['SpO2_median'], df['O2_any'], df['SBP_median'], df['HR_median'],
                     np.round(df['Cons_median']), df['Temp_median'])
df['NEWS2_0_24'] = t0; df['cat_0_24'] = news2_category(t0, r0)

# ── display names, permutation groups and colour domains ──
LABN = {'lactate': 'Lactate', 'creatinine': 'Creatinine', 'bun': 'Urea nitrogen', 'wbc': 'WBC', 'platelets': 'Platelets',
        'bilirubin': 'Bilirubin', 'hemoglobin': 'Haemoglobin', 'bicarbonate': 'Bicarbonate', 'potassium': 'Potassium',
        'glucose': 'Glucose', 'sodium': 'Sodium'}
VN = {'RR': 'RR', 'SpO2': 'SpO₂', 'HR': 'HR', 'SBP': 'SBP', 'Temp': 'Temp', 'MAP': 'MAP'}
INFO = {}
for f in FEATURES:
    if f.startswith('lab_'):
        n = f[4:]; INFO[f] = (f"{LABN[n]} ({'max' if LAB_DEF and dict((v[0], v[1]) for v in LAB_DEF.values())[n] == 'max' else 'min'})", 'Laboratory', 'Laboratory')
    elif f == 'lactate_measured': INFO[f] = ('Lactate measured', 'Laboratory', 'Laboratory')
    elif f.split('_')[0] in VN:
        v, st_ = f.split('_', 1); INFO[f] = (f"{VN[v]} {st_.replace('delta', 'change')}", 'Vital signs', v)
    elif f.startswith('GCS_'): INFO[f] = (f"GCS {f.split('_')[1]} (min)", 'Neurological & O₂', 'GCS & consciousness')
    elif f in ('Cons_worst', 'Cons_last'): INFO[f] = (f"Consciousness ({f.split('_')[1]})", 'Neurological & O₂', 'GCS & consciousness')
    elif f == 'O2_any': INFO[f] = ('Supplemental O₂', 'Neurological & O₂', 'Oxygen & ventilation')
    elif f == 'NEWS2_0_24': INFO[f] = ('NEWS2 (0–24 h)', 'NEWS2', 'NEWS2 (0–24 h)')
    elif f in TRT_F: INFO[f] = ({'vent_invasive': 'Invasive ventilation', 'vent_noninvasive': 'Non-invasive ventilation',
                                'vasopressor': 'Vasopressor'}[f], 'Treatment',
                               'Oxygen & ventilation' if f.startswith('vent') else 'Vasopressor')
    else: INFO[f] = ({'age': 'Age', 'male': 'Male sex', 'emergency_admission': 'Emergency admission'}.get(f, f.replace('unit_', 'Unit: ').capitalize()),
                     'Context', 'Demographics & unit')
def nice(f): return INFO[f][0]
GROUP = {f: INFO[f][2] for f in FEATURES}
DOMAIN = {f: INFO[f][1] for f in FEATURES}

tr, te = df[df.split == 'train'].reset_index(drop=True), df[df.split == 'test'].reset_index(drop=True)
X_tr, y_tr = tr[FEATURES].values.astype(float), tr['y'].values
X_te, y_te = te[FEATURES].values.astype(float), te['y'].values
scaler = StandardScaler().fit(X_tr)
X_tr_s, X_te_s = scaler.transform(X_tr), scaler.transform(X_te)
# FT-Transformer input: quantile-normal transform of continuous features (Gorishniy et al., 2021), standardised flags;
# both fitted on training data only. Skewed labs (lactate, creatinine, bilirubin …) become well-scaled for the network.
CONT = np.array([len(np.unique(X_tr[:, j])) > 10 for j in range(X_tr.shape[1])])
qt = QuantileTransformer(output_distribution='normal', n_quantiles=min(1000, len(X_tr)), subsample=10**9,
                         random_state=SEED).fit(X_tr[:, CONT])
def to_q(X):
    Z = scaler.transform(X); Z[:, CONT] = qt.transform(X[:, CONT]); return Z.astype(np.float32)
X_tr_q, X_te_q = to_q(X_tr), to_q(X_te)
cw = compute_class_weight('balanced', classes=np.arange(4), y=y_tr)
# patient-grouped 10% validation split of the training set (early stopping and FT-Transformer checkpointing)
fit_i, val_i = next(GroupShuffleSplit(n_splits=1, test_size=0.1, random_state=SEED).split(X_tr_s, y_tr, tr['subject_id']))
print(f'{len(FEATURES)} features | train {X_tr_s.shape} | test {X_te_s.shape} | class weights {np.round(cw, 3)}')
print('feature domains:', pd.Series(DOMAIN).value_counts().to_dict())
print(f'0–24 h category equals 24–48 h outcome for {(df.cat_0_24 == df.y).mean()*100:.1f}% of stays')


## 8. Model engine (shared with `tune_hyperparameters.ipynb`)
All learned models are fitted on the same 90% *fit* subset of the training patients; the remaining 10% (validation,
grouped by patient) is used for hyperparameter tuning, boosting early stopping, FT-Transformer checkpointing and the
choice of decision thresholds. The test patients are used only for the final evaluation.

**FT-Transformer** (Gorishniy et al., 2021): PLR periodic numerical embeddings (Gorishniy et al., 2022) on
quantile-normalised inputs, [CLS] token, pre-norm Transformer blocks with ReGLU feed-forward layers,
LayerNorm → ReLU → linear head; AdamW; class-weighted cross-entropy; early stopping on validation macro-F1.

In [ ]:
class LinearEmb(nn.Module):
    def __init__(s, n, d):
        super().__init__(); s.W = nn.Parameter(torch.empty(n, d)); s.b = nn.Parameter(torch.empty(n, d))
        for p in (s.W, s.b): nn.init.uniform_(p, -d ** -0.5, d ** -0.5)
    def forward(s, x): return x.unsqueeze(-1) * s.W + s.b
class PLREmb(nn.Module):                          # periodic → linear → ReLU, separate parameters per feature
    def __init__(s, n, d, k=48, sigma=0.01):
        super().__init__(); s.freq = nn.Parameter(torch.randn(n, k) * sigma)
        s.W = nn.Parameter(torch.empty(n, 2 * k, d)); s.b = nn.Parameter(torch.empty(n, d))
        for p in (s.W, s.b): nn.init.uniform_(p, -(2 * k) ** -0.5, (2 * k) ** -0.5)
    def forward(s, x):
        v = 2 * math.pi * s.freq[None] * x[..., None]
        v = torch.cat([torch.cos(v), torch.sin(v)], -1)
        return torch.relu(torch.einsum('bnk,nkd->bnd', v, s.W) + s.b)
class ReGLU(nn.Module):
    def forward(s, x): a, b = x.chunk(2, -1); return a * torch.relu(b)
class Block(nn.Module):                           # pre-norm block; the first block has no attention pre-norm (reference)
    def __init__(s, d, heads, ffn, att_drop, ffn_drop, res_drop, first):
        super().__init__(); s.n1 = nn.Identity() if first else nn.LayerNorm(d)
        s.att = nn.MultiheadAttention(d, heads, dropout=att_drop, batch_first=True)
        s.n2 = nn.LayerNorm(d)
        s.ff = nn.Sequential(nn.Linear(d, 2 * ffn), ReGLU(), nn.Dropout(ffn_drop), nn.Linear(ffn, d))
        s.rd = nn.Dropout(res_drop)
    def forward(s, x):
        h = s.n1(x); x = x + s.rd(s.att(h, h, h, need_weights=False)[0])
        return x + s.rd(s.ff(s.n2(x)))
class FTTransformer(nn.Module):
    def __init__(s, n, k, d=192, layers=3, ffn=256, att_drop=0.2, ffn_drop=0.1, res_drop=0.0, n_freq=48, sigma=0.01, heads=8, emb='plr'):
        super().__init__()
        s.tok = PLREmb(n, d, n_freq, sigma) if emb == 'plr' else LinearEmb(n, d)
        s.cls = nn.Parameter(torch.empty(1, 1, d)); nn.init.uniform_(s.cls, -d ** -0.5, d ** -0.5)
        s.blocks = nn.ModuleList([Block(d, heads, ffn, att_drop, ffn_drop, res_drop, i == 0) for i in range(layers)])
        s.head = nn.Sequential(nn.LayerNorm(d), nn.ReLU(), nn.Linear(d, k))
    def forward(s, x):
        t = torch.cat([s.cls.expand(x.shape[0], -1, -1), s.tok(x)], 1)
        for b in s.blocks: t = b(t)
        return s.head(t[:, 0])

FTT_KEYS = ['d', 'layers', 'ffn', 'att_drop', 'ffn_drop', 'res_drop', 'n_freq', 'sigma']
T = lambda a, dt=torch.float32: torch.as_tensor(np.asarray(a), dtype=dt).to(device)
@torch.no_grad()
def proba_of(net, X, bs=8192):
    net.eval(); return np.concatenate([torch.softmax(net(T(X[i:i+bs])), 1).cpu().numpy() for i in range(0, len(X), bs)])
def ens_proba(nets, X): return np.mean([proba_of(m, X) for m in nets], 0)

def train_ftt(Xf, yf, Xv, yv, K, P, class_w, seed, max_epochs=None, patience=None, trial=None, log_every=10):
    max_epochs, patience = max_epochs or FTT_EPOCHS, patience or FTT_PATIENCE
    torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)
    net = FTTransformer(Xf.shape[1], K, **{k: P[k] for k in FTT_KEYS}, emb=FTT_EMB).to(device)
    opt = optim.AdamW(net.parameters(), lr=P['lr'], weight_decay=P['wd'])
    crit = nn.CrossEntropyLoss(weight=T(class_w) if class_w is not None else None)
    dl = DataLoader(TensorDataset(T(Xf), T(yf, torch.long)), batch_size=256, shuffle=True,
                    generator=torch.Generator().manual_seed(seed))
    best, wait, hist = (-1, None, 0), 0, []
    for ep in range(1, max_epochs + 1):
        net.train(); tot = 0
        for xb, yb in dl:
            opt.zero_grad(); loss = crit(net(xb), yb); loss.backward(); opt.step(); tot += loss.item()
        vf1 = f1_score(yv, proba_of(net, Xv).argmax(1), average='macro')
        hist.append({'seed': seed, 'epoch': ep, 'train_loss': tot / len(dl), 'val_macro_f1': vf1})
        if vf1 > best[0]: best, wait = (vf1, copy.deepcopy(net.state_dict()), ep), 0
        else: wait += 1
        if log_every and ep % log_every == 0: print(f'    seed {seed} epoch {ep:3d} | loss {tot/len(dl):.4f} | val macro-F1 {vf1:.4f}')
        if trial is not None:
            trial.report(vf1, ep)
            if trial.should_prune(): raise optuna.TrialPruned()
        if wait >= patience: break
    net.load_state_dict(best[1]); net.eval()
    return net, best[2], best[0], hist, ep

def xgb_device():
    import xgboost
    if not torch.cuda.is_available(): return {'tree_method': 'hist'}
    return {'tree_method': 'hist', 'device': 'cuda'} if int(xgboost.__version__.split('.')[0]) >= 2 else {'tree_method': 'gpu_hist'}
XGB_FIXED = dict(n_estimators=3000, eval_metric='mlogloss', early_stopping_rounds=100, random_state=SEED, verbosity=0, n_jobs=-1, **xgb_device())
LGBM_FIXED = dict(n_estimators=3000, subsample_freq=1, random_state=SEED, verbose=-1)
def make_models(P, weighted=True):
    bal = 'balanced' if weighted else None
    return {'Logistic regression': LogisticRegression(max_iter=3000, class_weight=bal, random_state=SEED),
            'Random forest': RandomForestClassifier(**P['rf'], class_weight=bal, random_state=SEED, n_jobs=-1),
            'XGBoost': XGBClassifier(**P['xgb'], **XGB_FIXED),
            'LightGBM': LGBMClassifier(**P['lgbm'], **LGBM_FIXED, class_weight=bal)}
def fit_model(name, m, Xs_tr, ytr, fit, val, weighted=True):
    if name == 'XGBoost':
        sw = compute_sample_weight('balanced', ytr[fit]) if weighted else None
        m.fit(Xs_tr[fit], ytr[fit], sample_weight=sw, eval_set=[(Xs_tr[val], ytr[val])], verbose=False)
        return int(m.best_iteration) + 1
    if name == 'LightGBM':
        m.fit(Xs_tr[fit], ytr[fit], eval_set=[(Xs_tr[val], ytr[val])], callbacks=[lgbm.early_stopping(100, verbose=False)])
        return int(m.best_iteration_)
    m.fit(Xs_tr[fit], ytr[fit]); return None

def evaluate(name, y, pred, proba=None, K=4):
    try: au = roc_auc_score(y, proba, multi_class='ovr', average='macro') if proba is not None else np.nan
    except ValueError: au = np.nan
    return {'Model': name, 'Accuracy': accuracy_score(y, pred), 'Balanced Acc': balanced_accuracy_score(y, pred),
            'Macro-F1': f1_score(y, pred, average='macro', zero_division=0),
            'Weighted-F1': f1_score(y, pred, average='weighted', zero_division=0), 'Macro-AUROC': au,
            'High recall': recall_score(y, pred, labels=[K - 1], average='macro', zero_division=0),
            'Brier': np.mean(np.sum((proba - np.eye(K)[y]) ** 2, axis=1)) if proba is not None else np.nan}

def run_models(Xs_tr, Xs_te, Xq_tr, Xq_te, ytr, yte, K, weighted, n_seeds, fit, val, pers=None, verbose=True):
    # Fit every comparator and the FT-Transformer seed ensemble on one task; returns predictions and metrics.
    R = {'store': {}, 'val': {}, 'results': [], 'models': {}, 'best_iter': {}, 'hist': [], 'seeds': []}
    cwv = compute_class_weight('balanced', classes=np.arange(K), y=ytr[fit]) if weighted else None
    if pers is not None:
        R['store']['NEWS2 persistence'] = (pers, None); R['results'].append(evaluate('NEWS2 persistence', yte, pers, None, K))
    for name, m in make_models(PARAMS, weighted).items():
        t0 = time.time(); it = fit_model(name, m, Xs_tr, ytr, fit, val, weighted)
        pr = m.predict_proba(Xs_te); R['store'][name] = (pr.argmax(1), pr); R['val'][name] = m.predict_proba(Xs_tr[val])
        R['models'][name] = m; R['results'].append(evaluate(name, yte, pr.argmax(1), pr, K))
        if it is not None: R['best_iter'][name] = it
        if verbose: print(f"  ▶ {name:20s} macro-F1 {R['results'][-1]['Macro-F1']:.4f} | AUROC {R['results'][-1]['Macro-AUROC']:.4f} | "
                          f"accuracy {R['results'][-1]['Accuracy']:.4f}" + (f' | trees {it}' if it else '') + f' | {time.time()-t0:.0f}s')
    nets = []
    for si in range(n_seeds):
        t0 = time.time()
        net, bep, bf1, h, ep = train_ftt(Xq_tr[fit], ytr[fit], Xq_tr[val], ytr[val], K, PARAMS['ftt'], cwv, SEED + si,
                                         log_every=10 if verbose else 0)
        nets.append(net); R['hist'] += h; pt = proba_of(net, Xq_te)
        R['seeds'].append({'seed': SEED + si, 'best epoch': bep, 'epochs run': ep, 'val macro-F1': bf1,
                           'test macro-F1 (this seed)': f1_score(yte, pt.argmax(1), average='macro'),
                           'test accuracy (this seed)': accuracy_score(yte, pt.argmax(1)),
                           'minutes': (time.time() - t0) / 60})
        if verbose: print(f"  ▶ FT-Transformer seed {SEED+si}: best epoch {bep} of {ep} | val macro-F1 {bf1:.4f} | "
                          f"test macro-F1 {R['seeds'][-1]['test macro-F1 (this seed)']:.4f} | {R['seeds'][-1]['minutes']:.1f} min")
    pr = ens_proba(nets, Xq_te); R['store']['FT-Transformer'] = (pr.argmax(1), pr); R['val']['FT-Transformer'] = ens_proba(nets, Xq_tr[val])
    R['nets'] = nets; R['n_params'] = sum(p.numel() for p in nets[0].parameters())
    R['results'].append(evaluate('FT-Transformer', yte, pr.argmax(1), pr, K))
    R['res'] = pd.DataFrame(R['results']).sort_values('Macro-F1', ascending=False).reset_index(drop=True)
    return R

# Published default settings (used when no tuning output is attached)
DEFAULT_PARAMS = {
    'ftt':  dict(d=192, layers=3, ffn=256, att_drop=0.2, ffn_drop=0.1, res_drop=0.0, n_freq=48, sigma=0.01, lr=1e-4, wd=1e-5),
    'xgb':  dict(max_depth=6, learning_rate=0.03, subsample=0.8, colsample_bytree=0.8, min_child_weight=5),
    'lgbm': dict(num_leaves=63, learning_rate=0.03, subsample=0.8, colsample_bytree=0.8, min_child_samples=40),
    'rf':   dict(n_estimators=300, max_depth=14, min_samples_leaf=5, max_features='sqrt')}
PARAMS = copy.deepcopy(DEFAULT_PARAMS)
print('engine ready | XGBoost:', xgb_device())


## T1. Studies

In [ ]:
try:
    import optuna
except ImportError:
    import subprocess, sys; subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'optuna'], check=True); import optuna
import json, datetime
optuna.logging.set_verbosity(optuna.logging.WARNING)
E = lambda k, d: int(os.environ.get(k, d))
BUDGET = {'rf': (E('TUNE_TRIALS_RF', 30), E('TUNE_MIN_RF', 30)), 'xgb': (E('TUNE_TRIALS_XGB', 100), E('TUNE_MIN_XGB', 60)),
          'lgbm': (E('TUNE_TRIALS_LGBM', 100), E('TUNE_MIN_LGBM', 60)), 'ftt': (E('TUNE_TRIALS_FTT', 40), E('TUNE_MIN_FTT', 180))}
FTT_EPOCHS_TUNE, FTT_PATIENCE_TUNE = E('FTT_EPOCHS_TUNE', 100), E('FTT_PATIENCE_TUNE', 12)
Xf_s, Xv_s, Xf_q, Xv_q, yf, yv = X_tr_s[fit_i], X_tr_s[val_i], X_tr_q[fit_i], X_tr_q[val_i], y_tr[fit_i], y_tr[val_i]
cw_fit = compute_class_weight('balanced', classes=np.arange(4), y=yf)
vf1 = lambda proba: f1_score(yv, proba.argmax(1), average='macro')
MF = {'sqrt': 'sqrt', '0.3': 0.3, '0.5': 0.5, '0.7': 0.7}; MD = {'8': 8, '12': 12, '14': 14, '16': 16, '24': 24, '32': 32, 'none': None}

def p_rf(t):   return dict(n_estimators=t.suggest_int('n_estimators', 200, 800, step=100), max_depth=MD[t.suggest_categorical('max_depth', list(MD))],
                           min_samples_leaf=t.suggest_int('min_samples_leaf', 1, 30, log=True), max_features=MF[t.suggest_categorical('max_features', list(MF))])
def p_xgb(t):  return dict(max_depth=t.suggest_int('max_depth', 3, 10), learning_rate=t.suggest_float('learning_rate', 0.01, 0.2, log=True),
                           min_child_weight=t.suggest_float('min_child_weight', 1, 30, log=True), subsample=t.suggest_float('subsample', 0.5, 1.0),
                           colsample_bytree=t.suggest_float('colsample_bytree', 0.4, 1.0), gamma=t.suggest_float('gamma', 0.0, 5.0),
                           reg_lambda=t.suggest_float('reg_lambda', 1e-3, 10, log=True), reg_alpha=t.suggest_float('reg_alpha', 1e-3, 10, log=True))
def p_lgbm(t): return dict(num_leaves=t.suggest_int('num_leaves', 15, 255, log=True), learning_rate=t.suggest_float('learning_rate', 0.01, 0.2, log=True),
                           min_child_samples=t.suggest_int('min_child_samples', 5, 200, log=True), subsample=t.suggest_float('subsample', 0.5, 1.0),
                           colsample_bytree=t.suggest_float('colsample_bytree', 0.4, 1.0), reg_lambda=t.suggest_float('reg_lambda', 1e-3, 10, log=True),
                           reg_alpha=t.suggest_float('reg_alpha', 1e-3, 10, log=True))
def p_ftt(t):
    d = t.suggest_categorical('d', [64, 96, 128, 192, 256])
    return dict(d=d, layers=t.suggest_int('layers', 1, 4), ffn=int(d * t.suggest_float('ffn_factor', 2 / 3, 8 / 3)),
                att_drop=t.suggest_float('att_drop', 0.0, 0.5), ffn_drop=t.suggest_float('ffn_drop', 0.0, 0.5),
                res_drop=t.suggest_float('res_drop', 0.0, 0.2), n_freq=t.suggest_int('n_freq', 16, 96, step=16),
                sigma=t.suggest_float('sigma', 0.01, 10, log=True), lr=t.suggest_float('lr', 1e-5, 1e-3, log=True),
                wd=t.suggest_float('wd', 1e-6, 1e-3, log=True))
SPACE = {'Random forest': 'n_estimators 200–800 (step 100); max_depth {8, 12, 14, 16, 24, 32, none}; min_samples_leaf 1–30 (log); max_features {sqrt, 0.3, 0.5, 0.7}',
         'XGBoost': 'max_depth 3–10; learning_rate 0.01–0.2 (log); min_child_weight 1–30 (log); subsample 0.5–1; colsample_bytree 0.4–1; gamma 0–5; reg_lambda, reg_alpha 1e-3–10 (log); trees by early stopping (≤3,000)',
         'LightGBM': 'num_leaves 15–255 (log); learning_rate 0.01–0.2 (log); min_child_samples 5–200 (log); subsample 0.5–1; colsample_bytree 0.4–1; reg_lambda, reg_alpha 1e-3–10 (log); trees by early stopping (≤3,000)',
         'FT-Transformer': 'd {64, 96, 128, 192, 256}; blocks 1–4; FFN factor 2/3–8/3; attention dropout 0–0.5; FFN dropout 0–0.5; residual dropout 0–0.2; PLR frequencies 16–96 (step 16); σ 0.01–10 (log); lr 1e-5–1e-3 (log); weight decay 1e-6–1e-3 (log); batch 256; early stopping (patience 12, ≤100 epochs) + median pruner'}
DEFAULT_TRIAL = {'rf': {'n_estimators': 300, 'max_depth': '14', 'min_samples_leaf': 5, 'max_features': 'sqrt'},
                 'xgb': {'max_depth': 6, 'learning_rate': 0.03, 'min_child_weight': 5.0, 'subsample': 0.8, 'colsample_bytree': 0.8, 'gamma': 0.0, 'reg_lambda': 1.0, 'reg_alpha': 0.001},
                 'lgbm': {'num_leaves': 63, 'learning_rate': 0.03, 'min_child_samples': 40, 'subsample': 0.8, 'colsample_bytree': 0.8, 'reg_lambda': 0.001, 'reg_alpha': 0.001},
                 'ftt': {'d': 192, 'layers': 3, 'ffn_factor': 4 / 3, 'att_drop': 0.2, 'ffn_drop': 0.1, 'res_drop': 0.0, 'n_freq': 48, 'sigma': 0.01, 'lr': 1e-4, 'wd': 1e-5}}

def obj_rf(t):
    p = p_rf(t); m = RandomForestClassifier(**p, class_weight='balanced', random_state=SEED, n_jobs=-1).fit(Xf_s, yf)
    t.set_user_attr('P', p); return vf1(m.predict_proba(Xv_s))
def obj_xgb(t):
    p = p_xgb(t); m = XGBClassifier(**p, **XGB_FIXED)
    m.fit(Xf_s, yf, sample_weight=compute_sample_weight('balanced', yf), eval_set=[(Xv_s, yv)], verbose=False)
    t.set_user_attr('P', p); t.set_user_attr('trees', int(m.best_iteration) + 1); return vf1(m.predict_proba(Xv_s))
def obj_lgbm(t):
    p = p_lgbm(t); m = LGBMClassifier(**p, **LGBM_FIXED, class_weight='balanced')
    m.fit(Xf_s, yf, eval_set=[(Xv_s, yv)], callbacks=[lgbm.early_stopping(100, verbose=False)])
    t.set_user_attr('P', p); t.set_user_attr('trees', int(m.best_iteration_)); return vf1(m.predict_proba(Xv_s))
def obj_ftt(t):
    p = p_ftt(t); t.set_user_attr('P', p)
    net, ep, f1, _, _ = train_ftt(Xf_q, yf, Xv_q, yv, 4, p, cw_fit, SEED, FTT_EPOCHS_TUNE, FTT_PATIENCE_TUNE, trial=t, log_every=0)
    t.set_user_attr('best_epoch', ep); del net; torch.cuda.empty_cache() if torch.cuda.is_available() else None
    return f1

STUDIES, OUT = {}, {'features': FEATURES, 'n_fit': int(len(fit_i)), 'n_val': int(len(val_i)), 'objective': 'validation macro-F1 (4 categories, class-weighted)',
                    'n_trials': {}, 'best_val_macro_f1': {}, 'default_val_macro_f1': {}}
NAMES = {'rf': 'Random forest', 'xgb': 'XGBoost', 'lgbm': 'LightGBM', 'ftt': 'FT-Transformer'}
for key, obj in [('rf', obj_rf), ('xgb', obj_xgb), ('lgbm', obj_lgbm), ('ftt', obj_ftt)]:
    n_tr, mins = BUDGET[key]; t0 = time.time()
    st = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=SEED),
                             pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=10), study_name=key)
    st.enqueue_trial(DEFAULT_TRIAL[key])
    st.optimize(obj, n_trials=n_tr, timeout=mins * 60, gc_after_trial=True, catch=(ValueError, RuntimeError))
    STUDIES[key] = st; done = [t for t in st.trials if t.state == optuna.trial.TrialState.COMPLETE]
    OUT[key] = st.best_trial.user_attrs['P']; OUT['n_trials'][key] = len(done)
    OUT['best_val_macro_f1'][key] = st.best_value; OUT['default_val_macro_f1'][key] = st.trials[0].value
    if key in ('xgb', 'lgbm'): OUT[key + '_trees'] = st.best_trial.user_attrs.get('trees')
    st.trials_dataframe().to_csv(f'{FIG_DIR}/table_s5_trials_{key}.csv', index=False)
    json.dump(OUT, open(f'{FIG_DIR}/best_params.json', 'w'), indent=1, default=str)   # saved after every study
    print(f"▶ {NAMES[key]:15s} {len(done)} complete / {len(st.trials)} trials in {(time.time()-t0)/60:.1f} min | "
          f"val macro-F1 default {OUT['default_val_macro_f1'][key]:.4f} → best {st.best_value:.4f}")
    print('   ', OUT[key])
OUT['created'] = datetime.datetime.now().isoformat(timespec='seconds')
json.dump(OUT, open(f'{FIG_DIR}/best_params.json', 'w'), indent=1, default=str); print('✅ best_params.json')


## T2. Tuning summary (supplementary table and figure)

In [ ]:
summ = pd.DataFrame([{'Model': NAMES[k], 'Trials completed': OUT['n_trials'][k],
                      'Validation macro-F1, default': round(OUT['default_val_macro_f1'][k], 4),
                      'Validation macro-F1, tuned': round(OUT['best_val_macro_f1'][k], 4),
                      'Search space': SPACE[NAMES[k]], 'Selected': json.dumps(OUT[k], default=str)} for k in STUDIES])
save_table(summ, 'table_s4_tuning_summary'); print(summ.drop(columns=['Search space']).to_string(index=False))
fig, axes = plt.subplots(1, 4, figsize=(18, 3.8))
for ax, (k, st) in zip(axes, STUDIES.items()):
    v = np.array([t.value if t.state == optuna.trial.TrialState.COMPLETE else np.nan for t in st.trials])
    ax.scatter(np.arange(1, len(v) + 1), v, s=14, color='#b7d3f4', zorder=2, label='Trial')
    ax.plot(np.arange(1, len(v) + 1), pd.Series(v).cummax().ffill(), color=C_BLUE, lw=2, zorder=3, label='Best so far')
    ax.axhline(st.trials[0].value, color=C_GREY, ls='--', lw=1, label='Default setting')
    ax.set_title(NAMES[k], loc='left'); ax.set_xlabel('Trial'); ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True))
axes[0].set_ylabel('Validation macro-F1'); axes[0].legend(fontsize=7, loc='lower right')
plt.tight_layout(); save_fig(fig, 'fig_s2_tuning'); plt.show()
import zipfile
with zipfile.ZipFile(os.path.join(FIG_DIR, 'tuning_archive.zip'), 'w', zipfile.ZIP_DEFLATED) as z:
    for f in os.listdir(FIG_DIR):
        if f.endswith(('.png', '.pdf', '.csv', '.json')): z.write(os.path.join(FIG_DIR, f), f)
print('tuning_archive.zip written')
